# 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost
from xgboost import XGBRegressor
from xgboost import plot_importance
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import scipy.stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
plt.rc('font', family='NanumBarunGothic')
plt.rcParams['axes.unicode_minus'] =False

In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/BDA/data/모델링/train_mean.csv')
test_df = pd.read_csv('/content/drive/MyDrive/BDA/data/모델링/test_mean.csv')

In [ ]:
train = train_df.copy()
test = test_df.copy()

In [ ]:
train

In [ ]:
test

# 교통 인프라 지수 증가 시나리오 예측

교통 인프라 지수 하위 30% 이내 지역에 현재 인프라 지수에서 최댓값만큼 도달한다고 가정

In [ ]:
threshold_train = train['교통인프라지수'].quantile(0.3)

# 하위 30% 조건
low_infra_train = train['교통인프라지수'] <= threshold_train

# 방문 지역 추출
low_infra_regions_train = train.loc[low_infra_train, '방문 지역']
print(low_infra_regions_train.unique())  # 중복 제거해서 지역 목록 확인

교통 인프라 지수 증가 전

In [ ]:
train[train['방문 지역']==23310]['교통인프라지수']

In [ ]:
add = train['교통인프라지수'].max() - train['교통인프라지수']

In [ ]:
# --- train 처리 ---
threshold_train = train['교통인프라지수'].quantile(0.3)
low_infra_train = train['교통인프라지수'] <= threshold_train
train.loc[low_infra_train, '교통인프라지수'] += add

# --- test 처리 ---
threshold_test = test['교통인프라지수'].quantile(0.3)
low_infra_test = test['교통인프라지수'] <= threshold_test
test.loc[low_infra_test, '교통인프라지수'] += add

교통 인프라 지수 증가 후

In [ ]:
train[train['방문 지역']==23310]['교통인프라지수']

# 결측치 처리

결측치 모두 0으로 대체

In [ ]:
train = train.fillna(0)
test = test.fillna(0)

# 로그 변환

In [ ]:
cols = train.columns.tolist()[1:]

plt.figure(figsize=(15, 30))
for i, col in enumerate(cols):
    plt.subplot(len(cols), 2, 2*i + 1)
    sns.histplot(train[col], kde=True, bins=30, color='skyblue')
    plt.title(f'{col} (원본)')

    plt.subplot(len(cols), 2, 2*i + 2)
    sns.histplot(np.log1p(train[col]), kde=True, bins=30, color='salmon')
    plt.title(f'{col} (log 변환)')

plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = train.select_dtypes(include='number').drop(columns=['방문 지역'])
skewness = numeric_cols.skew().sort_values(ascending=False)

print("변수별 왜도:")
print(skewness)

In [ ]:
# for col in numeric_cols:
#     train[col] = np.log1p(train[col])
#     test[col] = np.log1p(test[col])

# numeric_cols = train.select_dtypes(include='number')
# skewness_after = numeric_cols.skew().sort_values(ascending=False)

# print("변수별 왜도:")
# print(skewness_after)

로그 변환 전/후 왜도 확인을 통해 유의미하게 왜도가 개선되는 변수만 로그 변환 진행

In [ ]:
log_cols = ['지출 비용', '철도비', '도로비', '쇼핑비', '숙박비', '여행 활동비', '교통인프라지수', '여행 일수', '자가용 의존도']

for col in log_cols:
    train[col] = np.log1p(train[col])
    test[col] = np.log1p(test[col])

# 스케일링

In [ ]:
cols = train.columns.tolist()[2:]

ss = StandardScaler()

train_scaled = train.copy()
test_scaled = test.copy()

train_scaled[cols] = ss.fit_transform(train[cols])
test_scaled[cols] = ss.transform(test[cols])

# LabelEncoding

In [ ]:
le = LabelEncoder()

train_scaled['방문 지역'] = le.fit_transform(train_scaled['방문 지역'])

test_scaled = test_scaled[test_scaled['방문 지역'].isin(le.classes_)]  # 알 수 없는 지역 제거
test_scaled['방문 지역'] = le.transform(test_scaled['방문 지역'])

# X, y 분리

In [ ]:
X = train_scaled.drop('지출 비용', axis=1)
y = train_scaled['지출 비용']

In [ ]:
test_ = test_scaled.drop('지출 비용', axis=1)
test_y = np.expm1(test_scaled['지출 비용'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=156)

# 모델링

## XGBoost

In [ ]:
xgb = XGBRegressor(n_estimators=1000, learning_rate=0.01,
                   max_depth=4, random_state=156)

xgb.fit(X_train, y_train)
pred = xgb.predict(X_test)

In [ ]:
mse = mean_squared_error(y_test, pred)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(f"RMSE: {np.sqrt(mse):.4f}")
print(f"MAE: {mae:.4f}")
print(f"R² (설명력): {r2:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
plot_importance(xgb,
                importance_type='gain',
                height=0.5,
                grid=False
                )

plt.title("Top 10 Feature Importances by Gain")
plt.tight_layout()
plt.show()

#### test 데이터 예측

In [ ]:
xgb.fit(X, y)
pred_test = xgb.predict(test_)
real_pred = np.expm1(pred_test)

In [ ]:
mse = mean_squared_error(test_y, real_pred)
mae = mean_absolute_error(test_y, real_pred)
r2 = r2_score(test_y, real_pred)

print(f"RMSE: {np.sqrt(mse):,.4f}")
print(f"MAE: {mae:,.4f}")
print(f"R^2 score: {r2:.4f}")

In [ ]:
print(f'test 데이터로 예측한 총 소비액: {real_pred.sum():,.4f}')
print(f'test 데이터의 실제 총 소비액: {test_y.sum():,.4f}')

|모델|총 소비액|
|-|-|
|기본 예측 모델 총 소비액|33,674,840원|
|교통 인프라 지수 증가 모델 총 소비액|33,745,904원|

-> 총 소비액이 증가했음을 확인

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=test_y, y=real_pred)
plt.plot([test_y.min(), test_y.max()], [test_y.min(), test_y.max()], 'r--')  # y=x 기준선
plt.xlabel("실제 소비액")
plt.ylabel("예측 소비액")
plt.title("실제값 vs 예측값 산점도")
# plt.xlim(0, 2*1e+8) # 이상치 제외 시각화
plt.grid(True)
plt.tight_layout()
plt.show()

#### 제출파일 생성

In [ ]:
sub = pd.DataFrame({
    '방문 지역': test_['방문 지역'],
    '예측 지출액': real_pred
})

In [ ]:
sub['방문 지역'] = le.inverse_transform(sub['방문 지역'])

In [ ]:
sub

In [ ]:
sub.to_csv('교통 인프라 지수 증가로 인한 총 예측 지출액.csv', index=False)